In [5]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    A token is already saved on your machine. Run `huggingface-cli whoami` to get more information or `huggingface-cli logout` if you want to log out.
    Setting a new token will erase the existing one.
    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) y
Token is valid (permission: fineG

In [2]:
!pip install bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 113.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 89.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 98.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalli

In [3]:
!git clone https://github.com/openai/human-eval
!pip install -e human-eval

Cloning into 'human-eval'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 34 (delta 12), reused 7 (delta 7), pack-reused 8 (from 1)
Receiving objects: 100% (34/34), 55.80 KiB | 27.90 MiB/s, done.
Resolving deltas: 100% (13/13), done.
Obtaining file:///content/human-eval
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 6.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for fire: filename=fire-0.7.0-py3-none-any.whl size=114249 sha256=e7fcbc69d5af97e8d38fa57d1e4059b5140a907cd23602b573bb2ad31cdf9eef
  Stored in directory: /root/.cache/pip/wheels/46/54/24/1624fd5b8674eb1188623f7e8e17cdf7c0f6c24b609dfb8a89
Successfully built fire
  Running setup.py develop for human-eval


In [2]:
cd human-eval

/content/human-eval


In [ ]:
from human_eval.data import read_problems, write_jsonl
import itertools
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

problems = read_problems()

model_name = "meta-llama/Llama-3.2-3B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token


# Apply 4-bit quantization
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type="nf4"
# )

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    #quantization_config=bnb_config,
    device_map="auto"
)

samples_first = []
samples_second = []
for problem_id in tqdm(problems.keys()):
    #  Prompt 1
    coding_prompt = problems[problem_id]["prompt"]
    prompt1_header = "You are an expert Python programmer, and here is your task: Complete the following python function: \n"
    prompt1 = prompt1_header + coding_prompt
    input1 = tokenizer(prompt1, return_tensors="pt").to(model.device)

    output1 = model.generate(
        input1.input_ids,
        max_length=500,
        temperature=0.2,
        top_p=0.95,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated_code1 = tokenizer.decode(output1[0], skip_special_tokens=True)

    completion1 = generated_code1[len(prompt1):] # remove header and coding prompt

    # Prompt 2
    prompt2_header = "There might be an error in the code below because of lack of understanding of the question. Please correct the error, if any, and rewrite the solution. Only output the final correct Python program! \n"
    prompt2 = prompt2_header + generated_code1[len(prompt1_header):] # adds just the coding portion of prompt 1's response

    input2 = tokenizer(prompt2, return_tensors="pt").to(model.device)

    output2 = model.generate(
        input2.input_ids,
        max_length=1000,
        temperature=0.2,
        top_p=0.95,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated_code2 = tokenizer.decode(output2[0], skip_special_tokens=True)

    completion2 = generated_code2[len(prompt2_header) + len(coding_prompt):] # remove header and coding prompt

    samples_first.append({
        "task_id": problem_id,
        "completion": completion1
    })

    samples_second.append({
        "task_id": problem_id,
        "completion": completion2
    })

    # print(f"Problem ID: {problem_id}")
    # print("=" * 40)
    # print("Prompt 1:")
    # print(prompt1)
    # print("=" * 40)
    # print("Prompt + Completion (First):")
    # print(generated_code1)
    # print("=" * 40)
    # print("Just Completion (First):")
    # print(completion1)
    # print("\n")
    # print("Prompt 2:")
    # print(prompt2)
    # print("=" * 40)
    # print("Prompt + Completion (Second):")
    # print(generated_code2)
    # print("=" * 40)
    # print("Just Completion (Second):")
    # print(completion2)
    # print("\n")

write_jsonl("humaneval_samples1.jsonl", samples_first)
write_jsonl("humaneval_samples2.jsonl", samples_second)

In [8]:
import gc
torch.cuda.empty_cache()
gc.collect()

59

In [ ]:
!evaluate_functional_correctness humaneval_samples1.jsonl

In [ ]:
!evaluate_functional_correctness humaneval_samples2.jsonl

In [15]:
import json

file_path = "humaneval_samples1.jsonl_results.jsonl"
data = []

with open(file_path, "r") as f:
    for line in f:
        data.append(json.loads(line.strip()))

data_dict1 = {i: item for i, item in enumerate(data)}

file_path = "humaneval_samples2.jsonl_results.jsonl"
data = []

with open(file_path, "r") as f:
    for line in f:
        data.append(json.loads(line.strip()))

data_dict2 = {i: item for i, item in enumerate(data)}

In [ ]:
# Get Results
count_i_c = 0
count_c_i = 0
correct_1 = 0
correct_2 = 0
num_problems = len(problems)
for i in range(len(data_dict1)):
    if data_dict1[i]['passed']:
        correct_1 += 1
    if data_dict2[i]['passed']:
        correct_2 += 1
    if data_dict1[i]['passed'] and not data_dict2[i]['passed']:
        count_c_i += 1
    elif not data_dict1[i]['passed'] and data_dict2[i]['passed']:
        count_i_c += 1

print("Accuracy@t1: " + str(correct_1 / num_problems))
print("Accuracy@t2: " + str(correct_2 / num_problems))
print("delta(t1,t2): " + str((correct_2 - correct_1) / num_problems))
print("delta(t1,t2) i to c: " + str(count_i_c / num_problems))
print("delta(t2,t1) c to i: " + str(count_c_i / num_problems))